# Compose an image transform pipeline

Image datasets apply a chain of small transforms to every sample. Write the transforms as callable classes and a `Compose` that runs them in order, the way `torchvision.transforms` is built.

## The steps
Each step is a tuple: a name followed by its arguments.

| Step | Effect |
|---|---|
| `("to_tensor",)` | uint8 image `(H, W, C)` to float64 `(C, H, W)` in `[0, 1]` (divide by 255) |
| `("normalize", mean, std)` | per channel: `(x - mean[c]) / std[c]`; `mean`, `std` are lists of length `C` |
| `("hflip",)` | mirror left-right (reverse the width axis) |
| `("center_crop", h, w)` | keep rows `top : top + h` and columns `left : left + w`, with `top = (H - h) // 2`, `left = (W - w) // 2` |
| `("pad", p, value)` | add `p` pixels of constant `value` on all four sides |

Every pipeline starts with `("to_tensor",)`; the rest come in any order and may repeat. All spatial steps act on the last two axes of a `(C, H, W)` tensor. Crops are never larger than the current image.

## The function that's graded
Write `apply_pipeline(img, steps)`:

- `img`: uint8 tensor `(H, W, C)`; `steps`: a list of step tuples.
- Build one transform object per step, `Compose` them, and return the result for `img` as a float64 `(C, H', W')` tensor.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

class Compose:
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, x):
        raise NotImplementedError

def apply_pipeline(img, steps):
    """img: uint8 (H, W, C). Return float64 (C, H', W') after every step."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[0, 255]], [[51, 102]]], "dtype": "uint8"}, [{"__tup__": ["to_tensor"]}, {"__tup__": ["hflip"]}]], "e": {"__nd__": [[[0.0], [0.2]], [[1.0], [0.4]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[255]]], "dtype": "uint8"}, [{"__tup__": ["to_tensor"]}, {"__tup__": ["normalize", [0.5], [0.25]]}, {"__tup__": ["pad", 1, 0.0]}]], "e": {"__nd__": [[[0.0, 0.0, 0.0], [0.0, 2.0, 0.0], [0.0, 0.0, 0.0]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[244, 99, 52], [184, 130, 116], [180, 83, 121], [147, 8, 60], [218, 44, 190], [252, 229, 123]], [[16, 45, 148], [8, 151, 81], [70, 63, 131], [164, 210, 87], [222, 201, 225], [81, 197, 163]], [[172, 137, 177], [222, 118, 9], [95, 12, 70], [34, 30, 100], [178, 61, 218], [146, 196, 6]], [[25, 112, 41], [56, 126, 99], [220, 120, 108], [95, 67, 16], [71, 10, 151], [79, 97, 27]], [[148, 138, 235], [138, 225, 122], [157, 122, 32], [210, 73, 245], [17, 65, 201], [61, 43, 27]]], "dtype": "uint8"}, [{"__tup__": ["to_tensor"]}, {"__tup__": ["center_crop", 3, 3]}, {"__tup__": ["normalize", [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]]}]], "e": {"__nd__": [[[-1.980905899477695, -0.9191711619145473, 0.6905556982618377], [1.6837914204983304, -0.49105231612295563, -1.5356622998544396], [-1.1589177155578387, 1.6495419128350033, -0.49105231612295563]], [[0.6078431372549019, -0.9327731092436975, 1.6407563025210081], [0.030112044817927192, -1.8256302521008403, -1.5105042016806725], [0.17016806722689076, 0.06512605042016796, -0.8627450980392157]], [[-0.3926797385620917, 0.4787799564270149, -0.28810457516339877], [-1.6475816993464054, -0.584400871459695, -0.061525054466231084], [-0.07895424836601315, 0.07790849673202599, -1.5255773420479304]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[96], [69], [212], [65]], [[165], [40], [154], [36]], [[59], [145], [74], [47]], [[241], [182], [214], [101]]], "dtype": "uint8"}, [{"__tup__": ["to_tensor"]}]], "e": {"__nd__": [[[0.3764705882352941, 0.27058823529411763, 0.8313725490196079, 0.2549019607843137], [0.6470588235294118, 0.1568627450980392, 0.6039215686274509, 0.1411764705882353], [0.23137254901960785, 0.5686274509803921, 0.2901960784313726, 0.1843137254901961], [0.9450980392156862, 0.7137254901960784, 0.8392156862745098, 0.396078431372549]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[105, 27], [161, 49], [90, 253], [75, 229], [155, 2]], [[87, 208], [136, 220], [247, 205], [22, 177], [72, 108]], [[40, 1], [199, 7], [152, 1], [132, 65], [187, 112]]], "dtype": "uint8"}, [{"__tup__": ["to_tensor"]}, {"__tup__": ["pad", 2, -1.0]}, {"__tup__": ["center_crop", 4, 6]}, {"__tup__": ["hflip"]}]], "e": {"__nd__": [[[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [0.6078431372549019, 0.29411764705882354, 0.35294117647058826, 0.6313725490196078, 0.4117647058823529, -1.0], [0.2823529411764706, 0.08627450980392157, 0.9686274509803922, 0.5333333333333333, 0.3411764705882353, -1.0], [0.7333333333333333, 0.5176470588235295, 0.596078431372549, 0.7803921568627451, 0.1568627450980392, -1.0]], [[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0], [0.00784313725490196, 0.8980392156862745, 0.9921568627450981, 0.19215686274509805, 0.10588235294117647, -1.0], [0.4235294117647059, 0.6941176470588235, 0.803921568627451, 0.8627450980392157, 0.8156862745098039, -1.0], [0.4392156862745098, 0.2549019607843137, 0.00392156862745098, 0.027450980392156862, 0.00392156862745098, -1.0]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[199, 104, 69], [151, 52, 123], [44, 115, 213], [150, 177, 154], [155, 79, 235], [96, 215, 124], [152, 165, 85]], [[18, 0, 7], [217, 166, 78], [233, 90, 64], [53, 137, 98], [232, 115, 38], [150, 227, 192], [208, 18, 58]], [[19, 95, 93], [12, 40, 224], [233, 229, 220], [72, 38, 201], [173, 228, 29], [212, 75, 4], [10, 55, 214]], [[110, 159, 81], [122, 46, 50], [225, 171, 111], [9, 0, 9], [101, 122, 176], [191, 252, 188], [2, 176, 100]], [[183, 93, 227], [193, 239, 225], [225, 157, 231], [68, 86, 99], [62, 189, 199], [115, 52, 21], [204, 19, 203]], [[5, 122, 155], [245, 97, 140], [101, 142, 117], [26, 57, 70], [213, 197, 41], [13, 251, 76], [34, 217, 90]]], "dtype": "uint8"}, [{"__tup__": ["to_tensor"]}, {"__tup__": ["hflip"]}, {"__tup__": ["hflip"]}, {"__tup__": ["center_crop", 6, 7]}]], "e": {"__nd__": [[[0.7803921568627451, 0.592156862745098, 0.17254901960784313, 0.5882352941176471, 0.6078431372549019, 0.3764705882352941, 0.596078431372549], [0.07058823529411765, 0.8509803921568627, 0.9137254901960784, 0.20784313725490197, 0.9098039215686274, 0.5882352941176471, 0.8156862745098039], [0.07450980392156863, 0.047058823529411764, 0.9137254901960784, 0.2823529411764706, 0.6784313725490196, 0.8313725490196079, 0.0392156862745098], [0.43137254901960786, 0.47843137254901963, 0.8823529411764706, 0.03529411764705882, 0.396078431372549, 0.7490196078431373, 0.00784313725490196], [0.7176470588235294, 0.7568627450980392, 0.8823529411764706, 0.26666666666666666, 0.24313725490196078, 0.45098039215686275, 0.8], [0.0196078431372549, 0.9607843137254902, 0.396078431372549, 0.10196078431372549, 0.8352941176470589, 0.050980392156862744, 0.13333333333333333]], [[0.40784313725490196, 0.20392156862745098, 0.45098039215686275, 0.6941176470588235, 0.30980392156862746, 0.8431372549019608, 0.6470588235294118], [0.0, 0.6509803921568628, 0.35294117647058826, 0.5372549019607843, 0.45098039215686275, 0.8901960784313725, 0.07058823529411765], [0.37254901960784315, 0.1568627450980392, 0.8980392156862745, 0.14901960784313725, 0.8941176470588236, 0.29411764705882354, 0.21568627450980393], [0.6235294117647059, 0.1803921568627451, 0.6705882352941176, 0.0, 0.47843137254901963, 0.9882352941176471, 0.6901960784313725], [0.36470588235294116, 0.9372549019607843, 0.615686274509804, 0.33725490196078434, 0.7411764705882353, 0.20392156862745098, 0.07450980392156863], [0.47843137254901963, 0.3803921568627451, 0.5568627450980392, 0.2235294117647059, 0.7725490196078432, 0.984313725490196, 0.8509803921568627]], [[0.27058823529411763, 0.4823529411764706, 0.8352941176470589, 0.6039215686274509, 0.9215686274509803, 0.48627450980392156, 0.3333333333333333], [0.027450980392156862, 0.3058823529411765, 0.25098039215686274, 0.3843137254901961, 0.14901960784313725, 0.7529411764705882, 0.22745098039215686], [0.36470588235294116, 0.8784313725490196, 0.8627450980392157, 0.788235294117647, 0.11372549019607843, 0.01568627450980392, 0.8392156862745098], [0.3176470588235294, 0.19607843137254902, 0.43529411764705883, 0.03529411764705882, 0.6901960784313725, 0.7372549019607844, 0.39215686274509803], [0.8901960784313725, 0.8823529411764706, 0.9058823529411765, 0.38823529411764707, 0.7803921568627451, 0.08235294117647059, 0.796078431372549], [0.6078431372549019, 0.5490196078431373, 0.4588235294117647, 0.27450980392156865, 0.1607843137254902, 0.2980392156862745, 0.35294117647058826]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[255, 78, 76], [84, 19, 213], [136, 223, 242], [143, 154, 24], [148, 68, 210], [17, 69, 175], [167, 93, 216], [180, 29, 247]], [[163, 79, 206], [117, 71, 39], [107, 179, 199], [163, 15, 184], [237, 227, 64], [157, 130, 87], [245, 221, 248], [131, 112, 83]], [[240, 25, 69], [13, 169, 132], [61, 38, 156], [179, 194, 244], [61, 217, 93], [26, 39, 145], [88, 184, 146], [157, 13, 115]], [[61, 68, 32], [253, 109, 77], [23, 243, 55], [45, 104, 30], [56, 52, 64], [3, 0, 109], [1, 96, 115], [79, 200, 106]], [[11, 184, 147], [128, 39, 252], [90, 44, 108], [9, 74, 191], [192, 198, 227], [136, 109, 185], [159, 79, 131], [11, 41, 29]], [[56, 237, 44], [115, 25, 209], [128, 231, 27], [97, 178, 70], [179, 159, 186], [77, 244, 124], [252, 114, 40], [86, 22, 191]], [[21, 168, 141], [130, 210, 125], [105, 157, 202], [171, 165, 144], [177, 77, 152], [119, 187, 215], [69, 180, 215], [132, 10, 224]], [[37, 234, 231], [61, 24, 146], [44, 181, 255], [210, 103, 161], [39, 112, 176], [53, 233, 209], [19, 23, 245], [216, 86, 184]]], "dtype": "uint8"}, [{"__tup__": ["to_tensor"]}, {"__tup__": ["center_crop", 5, 4]}, {"__tup__": ["pad", 1, 0.5]}, {"__tup__": ["normalize", [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]]}, {"__tup__": ["hflip"]}]], "e": {"__nd__": [[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.2313725490196079, 0.8588235294117648, 0.2784313725490195, -0.16078431372549018, 0.0], [0.0, -0.7960784313725491, -0.5215686274509803, 0.4039215686274509, -0.5215686274509803, 0.0], [0.0, -0.9764705882352941, -0.5607843137254902, -0.6470588235294117, -0.8196078431372549, 0.0], [0.0, 0.06666666666666665, 0.5058823529411764, -0.9294117647058824, -0.2941176470588235, 0.0], [0.0, -0.39607843137254906, 0.4039215686274509, -0.23921568627450984, 0.0039215686274509665, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.019607843137254832, 0.780392156862745, -0.8823529411764706, 0.4039215686274509, 0.0], [0.0, -0.6941176470588235, 0.7019607843137254, 0.5215686274509803, -0.7019607843137254, 0.0], [0.0, -1.0, -0.592156862745098, -0.1843137254901961, 0.9058823529411764, 0.0], [0.0, -0.14509803921568631, 0.5529411764705883, -0.41960784313725485, -0.6549019607843137, 0.0], [0.0, 0.9137254901960785, 0.24705882352941178, 0.39607843137254894, 0.8117647058823529, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -0.3176470588235294, -0.4980392156862745, 0.44313725490196076, 0.5607843137254902, 0.0], [0.0, 0.13725490196078427, -0.2705882352941177, 0.9137254901960785, 0.22352941176470598, 0.0], [0.0, -0.14509803921568631, -0.4980392156862745, -0.7647058823529411, -0.5686274509803921, 0.0], [0.0, 0.4509803921568627, 0.780392156862745, 0.4980392156862745, -0.15294117647058825, 0.0], [0.0, -0.027450980392156876, 0.45882352941176463, -0.4509803921568627, -0.788235294117647, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]], "dtype": "float64"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(apply_pipeline, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")